## Analyze Words in a Corpus
This script illustrates how we can use RUMLEM to count and analyse the words of a corpus.

### Preliminaries: Import and Setup

In [5]:
# Imports and setup
from rumlem import Lemmatizer
from collections import Counter, defaultdict
from datasets import load_dataset
import re
import gc
import pandas as pd


mediomatix = load_dataset("ZurichNLP/mediomatix", split="train[:5000]")
idioms = ["rm-sursilv", "rm-sutsilv", "rm-surmiran", "rm-puter", "rm-vallader"] # no "rm-rumgr" present


In [6]:
# Inspect the data
inspection_data = mediomatix[0]
for k, v in inspection_data.items():
    print(k, v)
print()

rm-sursilv Silsuenter emprovan ils scolars e las scolaras da leger il text en gruppas da dus. Per finir cantar la canzun el plenum (notas sils models M10.1a-d).
rm-sutsilv Suainter san igls sculars a las scularas ampruvar da liger igl text an grupas da dus a sco finiziùn cantar anzemel la canzùn (notas sen igls modèls M10.1a-d).
rm-surmiran None
rm-puter None
rm-vallader None
book 2.1_tc
chapter 10-hei-hou-in-e-dus-e-treis



### Building the Corpus
Iterate over the mediomatix corpus, clean up the data and save each sentence to a list. The list is the value, the idiom name the key of the dictionary.

In [7]:

# Build corpus
corpus = {k:[] for k in idioms}
for row in mediomatix:
    for idiom in idioms:
        if row.get(idiom):
            sent = row[idiom]
            sent = re.sub(r"</?strong>", " ", sent) # Remove <strong> tags
            sent = re.sub(r'["“”«»‚‘’„][^"“”«»‚‘’„]+["“”«»‚‘’„]', " ", sent) # Remove quoted substrings (straight, angled, curly quotes)
            corpus[idiom].append(sent)

for k, v in corpus.items():
    print(k, len(v))

rm-sursilv 4864
rm-sutsilv 4562
rm-surmiran 1036
rm-puter 4846
rm-vallader 4816


### Analyzing the Corpus
Instantiate the lemmatizers for each idiom, then iterate over our previously built corpus. For each relevant token (not punctuation), record its first lemma and keep track of the number of its occurences.

In [8]:
corpus_lemma_counter = {
    k: defaultdict(lambda: {"count": 0, "forms": Counter()}) for k in idioms
}

for idiom in idioms:
    lemmatizer = Lemmatizer(idiom=idiom)
    for sent in corpus[idiom]:
        doc = lemmatizer(sent)
        for t in doc.tokens:
            if any(c in t.text for c in ["/", ".", "-", ":", "(", ")"]):
                continue
            if not t.lemmas:
                continue
            first_lemma = next(iter(t.lemmas.keys()))
            lemma_text = first_lemma.text
            translation = first_lemma.translation_de or ""
            entry = corpus_lemma_counter[idiom][(lemma_text, translation)]
            entry["count"] += 1
            entry["forms"][t.text] += 1
    
    del lemmatizer
    gc.collect()

### Pretty Print Results
Finally, print the results in a nice format.

In [9]:
#  Pretty print results
for idiom, counter in corpus_lemma_counter.items():
   
    rows = []
    for (lemma_text, trans), data in sorted(counter.items(), key=lambda x: x[1]["count"], reverse=True)[:20]:
        forms_str = ", ".join(f"{form} ({count})" for form, count in data["forms"].items())
        rows.append((lemma_text, trans or "", data["count"], forms_str))
    
    df = pd.DataFrame(rows, columns=["Lemma", "German", "Freq", "Forms"])
    print(f"\n===== {idiom} =====")

    # Show full content of cells, but limit visible width (characters)
    pd.set_option('display.max_colwidth', 100)  # 100 chars per cell

    display(df)


===== rm-sursilv =====


,Lemma,German,Freq,Forms
0,il,cf. il,1658,"ils (495), il (781), Ils (165), Il (217)"
1,da,cf. da,958,"da (939), Da (17), DA (2)"
2,esser,sein,951,"ei (665), seigi (12), ein (124), fuss (12), eis (38), Ei (39), eran (18), essan (3), Eis (18), e..."
3,e,dritter Ton der Tonleiter in C-Dur,870,"e (832), E (38)"
4,in,eins,845,"in (778), In (67)"
5,la,La,776,"la (642), La (134)"
6,haver,besitzen,733,"giu (49), ha (367), haver (18), havein (14), han (105), havess (13), havevan (10), haveva (38), ..."
7,ina,eine,515,"ina (482), Ina (33)"
8,en,in,496,"en (457), En (39)"
9,che,dass,491,"che (485), Che (6)"



===== rm-sutsilv =====


,Lemma,German,Freq,Forms
0,igl,es,1502,"igl (1148), Igl (354)"
1,la,null,1253,"la (1062), La (191)"
2,a,"A, a",1073,"a (1018), A (55)"
3,da,von,1072,"da (1056), Da (14), DA (2)"
4,egn,Eins,725,"egn (663), Egn (62)"
5,easser,sich aufhalten,714,"segi (7), e (484), en (119), eara (8), fuss (9), E (15), earan (8), es (14), eassan (4), stos (6..."
6,ver,besitzen,707,"â (361), vaseva (3), vevan (5), veza (19), ve (38), ân (92), vieu (9), vegi (6), veva (13), vez ..."
7,igls,null,630,"igls (458), Igls (172)"
8,an,in,546,"an (495), An (51)"
9,las,null,545,"las (508), Las (37)"



===== rm-surmiran =====


,Lemma,German,Freq,Forms
0,igl,den,287,"igl (247), Igl (40)"
1,en,"ein, eine, ein",270,"ena (106), en (127), Ena (16), En (21)"
2,la,die,253,"la (211), La (42)"
3,da,als,216,"Da (4), da (212)"
4,esser,liegen,170,"è (115), èn (30), Fiss (1), Èn (1), sung (6), Ist (9), fiss (2), ist (2), È (1), ischan (1), Isc..."
5,igls,ihnen,152,"igls (141), Igls (11)"
6,veir,haben,149,"ò (76), ast (16), on (21), Ast (12), Vez (3), vegia (2), On (1), gia (6), vess (3), vez (2), va ..."
7,tgi,als,149,"tgi (123), Tgi (26)"
8,cun,anhand,146,"cun (145), Cun (1)"
9,las,ihnen,142,"las (112), Las (30)"



===== rm-puter =====


,Lemma,German,Freq,Forms
0,la,"cf. a, da, in, giò, sün + art.",1532,"las (508), la (852), La (127), Las (45)"
1,da,in (zeitlich),1198,"da (1181), Da (15), DA (2)"
2,il,das (vor Konsonant),1081,"il (834), Il (247)"
3,e,"E, e",885,"e (850), E (35)"
4,ün,Eins,819,"ün (755), Ün (64)"
5,avair,Lust haben,761,"ho (427), vains (10), he (26), haun (119), hegia (4), vaiva (30), vais (12), Hest (27), hest (55..."
6,a,bei (örtlich),729,"a (630), A (99)"
7,in,in (örtlich),628,"in (589), In (39)"
8,ils,die,609,"ils (424), Ils (185)"
9,esser,im Wald sein,606,"es (468), eira (40), füss (15), eiran (14), est (13), Eira (1), essans (3), Est (14), Es (8), es..."



===== rm-vallader =====


,Lemma,German,Freq,Forms
0,la,"cf. a, da, in, giò, sün + art.",1466,"la (817), las (475), La (131), Las (43)"
1,da,allermeiste,1194,"da (1180), Da (13), DA (1)"
2,il,"der, das, die",1053,"il (814), Il (239)"
3,e,"E, e",891,"e (857), E (34)"
4,ün,ein,813,"ün (749), Ün (64)"
5,a,bei (örtlich),650,"a (593), A (57)"
6,in,bei (zeitlich),609,"in (569), In (40)"
7,ils,die,577,"ils (392), Ils (185)"
8,esser,an etw gewöhnt sein,551,"es (411), eira (44), eiran (14), est (12), füss (13), saja (14), eschan (3), Est (14), eschat (6..."
9,üna,eine,525,"üna (499), Üna (26)"
